# Airbnb Seattle Data Cleaning & Exploratory Data Analysis

This notebook documents a reproducible data-cleaning and EDA workflow for the Seattle Airbnb project using the original `listings`, `reviews`, and `calendar` CSV datasets.

The workflow covers:
- Loading and understanding the datasets
- Removing duplicate and irrelevant metadata/URL columns
- Handling missing values where appropriate
- Converting price fields from text to numeric
- Converting date fields to datetime
- Standardizing boolean-style fields
- Creating calendar month/year/day features
- Performing basic exploratory analysis
- Exporting cleaned datasets for SQL and Power BI


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Load the Original Datasets

Place this notebook in the same folder as the three original CSV files, or update the file paths below.


In [3]:
listings = pd.read_csv("listings.csv", low_memory=False)
reviews = pd.read_csv("reviews.csv", low_memory=False)
calendar = pd.read_csv("calendar.csv", low_memory=False)

print("Listings shape:", listings.shape)
print("Reviews shape:", reviews.shape)
print("Calendar shape:", calendar.shape)


Listings shape: (3818, 92)
Reviews shape: (84849, 6)
Calendar shape: (1393570, 4)


## 2. Initial Data Understanding


In [4]:
print("\nLISTINGS")
display(listings.head())
listings.info()

print("\nREVIEWS")
display(reviews.head())
reviews.info()

print("\nCALENDAR")
display(calendar.head())
calendar.info()



LISTINGS


,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,notes,transit,thumbnail_url,medium_url,picture_url,xl_picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,street,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,city,state,zipcode,market,smart_location,country_code,country,latitude,longitude,is_location_exact,property_type,room_type,accommodates,bathrooms,bedrooms,beds,bed_type,amenities,square_feet,price,weekly_price,monthly_price,security_deposit,cleaning_fee,guests_included,extra_people,minimum_nights,maximum_nights,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,241032,https://www.airbnb.com/rooms/241032,20160104002432,2016-01-04,Stylish Queen Anne Apartment,NaN,Make your self at home in this charming one-be...,Make your self at home in this charming one-be...,none,NaN,NaN,NaN,NaN,NaN,https://a1.muscache.com/ac/pictures/67560560/c...,NaN,956883,https://www.airbnb.com/users/show/956883,Maija,2011-08-11,"Seattle, Washington, United States","I am an artist, interior designer, and run a s...",within a few hours,96%,100%,f,https://a0.muscache.com/ac/users/956883/profil...,https://a0.muscache.com/ac/users/956883/profil...,Queen Anne,3.0,3.0,"['email', 'phone', 'reviews', 'kba']",t,t,"Gilman Dr W, Seattle, WA 98119, United States",Queen Anne,West Queen Anne,Queen Anne,Seattle,WA,98119,Seattle,"Seattle, WA",US,United States,47.636289,-122.371025,t,Apartment,Entire home/apt,4,1.0,1.0,1.0,Real Bed,"{TV,""Cable TV"",Internet,""Wireless Internet"",""A...",NaN,$85.00,NaN,NaN,NaN,NaN,2,$5.00,1,365,4 weeks ago,t,14,41,71,346,2016-01-04,207,2011-11-01,2016-01-02,95.0,10.0,10.0,10.0,10.0,9.0,10.0,f,NaN,WASHINGTON,f,moderate,f,f,2,4.07
1,953595,https://www.airbnb.com/rooms/953595,20160104002432,2016-01-04,Bright & Airy Queen Anne Apartment,Chemically sensitive? We've removed the irrita...,"Beautiful, hypoallergenic apartment in an extr...",Chemically sensitive? We've removed the irrita...,none,"Queen Anne is a wonderful, truly functional vi...",What's up with the free pillows? Our home was...,"Convenient bus stops are just down the block, ...",https://a0.muscache.com/ac/pictures/14409893/f...,https://a0.muscache.com/im/pictures/14409893/f...,https://a0.muscache.com/ac/pictures/14409893/f...,https://a0.muscache.com/ac/pictures/14409893/f...,5177328,https://www.airbnb.com/users/show/5177328,Andrea,2013-02-21,"Seattle, Washington, United States",Living east coast/left coast/overseas. Time i...,within an hour,98%,100%,t,https://a0.muscache.com/ac/users/5177328/profi...,https://a0.muscache.com/ac/users/5177328/profi...,Queen Anne,6.0,6.0,"['email', 'phone', 'facebook', 'linkedin', 're...",t,t,"7th Avenue West, Seattle, WA 98119, United States",Queen Anne,West Queen Anne,Queen Anne,Seattle,WA,98119,Seattle,"Seattle, WA",US,United States,47.639123,-122.365666,t,Apartment,Entire home/apt,4,1.0,1.0,1.0,Real Bed,"{TV,Internet,""Wireless Internet"",Kitchen,""Free...",NaN,$150.00,"$1,000.00","$3,000.00",$100.00,$40.00,1,$0.00,2,90,today,t,13,13,16,291,2016-01-04,43,2013-08-19,2015-12-29,96.0,10.0,10.0,10.0,10.0,10.0,10.0,f,NaN,WASHINGTON,f,strict,t,t,6,1.48
2,3308979,https://www.airbnb.com/rooms/3308979,20160104002432,2016-01-04,New Modern House-Amazi

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3818 entries, 0 to 3817
Data columns (total 92 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                3818 non-null   int64  
 1   listing_url                       3818 non-null   object 
 2   scrape_id                         3818 non-null   int64  
 3   last_scraped                      3818 non-null   object 
 4   name                              3818 non-null   object 
 5   summary                           3641 non-null   object 
 6   space                             3249 non-null   object 
 7   description                       3818 non-null   object 
 8   experiences_offered               3818 non-null   object 
 9   neighborhood_overview             2786 non-null   object 
 10  notes                             2212 non-null   object 
 11  transit                           2884 non-null   object 
 12  thumbn

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,7202016,38917982,2015-07-19,28943674,Bianca,Cute and cozy place. Perfect location to every...
1,7202016,39087409,2015-07-20,32440555,Frank,Kelly has a great room in a very central locat...
2,7202016,39820030,2015-07-26,37722850,Ian,"Very spacious apartment, and in a great neighb..."
3,7202016,40813543,2015-08-02,33671805,George,Close to Seattle Center and all it has to offe...
4,7202016,41986501,2015-08-10,34959538,Ming,Kelly was a great host and very accommodating ...


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84849 entries, 0 to 84848
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   listing_id     84849 non-null  int64 
 1   id             84849 non-null  int64 
 2   date           84849 non-null  object
 3   reviewer_id    84849 non-null  int64 
 4   reviewer_name  84849 non-null  object
 5   comments       84831 non-null  object
dtypes: int64(3), object(3)
memory usage: 3.9+ MB

CALENDAR


,listing_id,date,available,price
0,241032,2016-01-04,t,$85.00
1,241032,2016-01-05,t,$85.00
2,241032,2016-01-06,f,NaN
3,241032,2016-01-07,f,NaN
4,241032,2016-01-08,f,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1393570 entries, 0 to 1393569
Data columns (total 4 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   listing_id  1393570 non-null  int64 
 1   date        1393570 non-null  object
 2   available   1393570 non-null  object
 3   price       934542 non-null   object
dtypes: int64(1), object(3)
memory usage: 42.5+ MB


In [5]:
summary = pd.DataFrame({
    "Dataset": ["Listings", "Reviews", "Calendar"],
    "Rows": [len(listings), len(reviews), len(calendar)],
    "Columns": [listings.shape[1], reviews.shape[1], calendar.shape[1]],
    "Duplicate Rows": [
        listings.duplicated().sum(),
        reviews.duplicated().sum(),
        calendar.duplicated().sum()
    ]
})
display(summary)


,Dataset,Rows,Columns,Duplicate Rows
0,Listings,3818,92,0
1,Reviews,84849,6,0
2,Calendar,1393570,4,0


## 3. Clean Listings Dataset

URL, image URL, and scrape-related metadata columns are removed because they are identifiers or web/scraping metadata and are not required for the project's pricing, availability, neighbourhood, room-type, review, rating, and host analysis.

Columns that are entirely empty are also removed. Duplicate rows are removed, while important listing-level fields are retained for analysis.


In [6]:
listings_clean = listings.copy()

irrelevant_listing_columns = [
    "listing_url", "scrape_id", "last_scraped",
    "thumbnail_url", "medium_url", "picture_url", "xl_picture_url",
    "host_url", "host_thumbnail_url", "host_picture_url",
    "calendar_last_scraped"
]

listings_clean = listings_clean.drop(
    columns=[c for c in irrelevant_listing_columns if c in listings_clean.columns]
)

# Remove columns containing no usable values
listings_clean = listings_clean.dropna(axis=1, how="all")

# Remove exact duplicate rows
listings_clean = listings_clean.drop_duplicates()

print("Cleaned listings shape:", listings_clean.shape)


Cleaned listings shape: (3818, 80)


### Convert currency fields to numeric


In [10]:
money_columns = [
    "price", "weekly_price", "monthly_price",
    "security_deposit", "cleaning_fee", "extra_people"
]

for col in money_columns:
    if col in listings_clean.columns:
        listings_clean[col] = (
            listings_clean[col]
            .astype(str)
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .replace("nan", np.nan)
        )
        listings_clean[col] = pd.to_numeric(listings_clean[col], errors="coerce")


### Convert percentage and date fields


In [9]:
percentage_columns = ["host_response_rate", "host_acceptance_rate"]

for col in percentage_columns:
    if col in listings_clean.columns:
        listings_clean[col] = (
            listings_clean[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .replace("nan", np.nan)
        )
        listings_clean[col] = pd.to_numeric(listings_clean[col], errors="coerce")

listing_date_columns = ["host_since", "first_review", "last_review"]

for col in listing_date_columns:
    if col in listings_clean.columns:
        listings_clean[col] = pd.to_datetime(listings_clean[col], errors="coerce")


### Standardize selected true/false fields


In [ ]:
boolean_columns = [
    "host_is_superhost", "host_has_profile_pic", "host_identity_verified",
    "is_location_exact", "has_availability", "requires_license",
    "instant_bookable", "require_guest_profile_picture",
    "require_guest_phone_verification"
]

for col in boolean_columns:
    if col in listings_clean.columns:
        listings_clean[col] = (
            listings_clean[col]
            .astype(str)
            .str.strip()
            .str.lower()
        )


## 4. Clean Reviews Dataset


In [11]:
reviews_clean = reviews.copy()

reviews_clean = reviews_clean.drop_duplicates()

# listing_id and review id are essential identifiers
reviews_clean = reviews_clean.dropna(subset=["listing_id", "id"])

reviews_clean["date"] = pd.to_datetime(reviews_clean["date"], errors="coerce")

print("Cleaned reviews shape:", reviews_clean.shape)
display(reviews_clean.head())


Cleaned reviews shape: (84849, 6)


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,7202016,38917982,2015-07-19,28943674,Bianca,Cute and cozy place. Perfect location to every...
1,7202016,39087409,2015-07-20,32440555,Frank,Kelly has a great room in a very central locat...
2,7202016,39820030,2015-07-26,37722850,Ian,"Very spacious apartment, and in a great neighb..."
3,7202016,40813543,2015-08-02,33671805,George,Close to Seattle Center and all it has to offe...
4,7202016,41986501,2015-08-10,34959538,Ming,Kelly was a great host and very accommodating ...


## 5. Clean Calendar Dataset

The calendar price field is stored as text with currency symbols, so it is converted to numeric. Date-derived columns are created for monthly, yearly, and day-of-week analysis in Power BI.


In [12]:
calendar_clean = calendar.copy()

calendar_clean = calendar_clean.drop_duplicates()
calendar_clean = calendar_clean.dropna(subset=["listing_id", "date"])

calendar_clean["date"] = pd.to_datetime(calendar_clean["date"], errors="coerce")

calendar_clean["available"] = (
    calendar_clean["available"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"t": True, "f": False})
)

calendar_clean["price"] = (
    calendar_clean["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .replace("nan", np.nan)
)
calendar_clean["price"] = pd.to_numeric(calendar_clean["price"], errors="coerce")

calendar_clean["month"] = calendar_clean["date"].dt.month
calendar_clean["month_name"] = calendar_clean["date"].dt.month_name()
calendar_clean["year"] = calendar_clean["date"].dt.year
calendar_clean["day_of_week"] = calendar_clean["date"].dt.day_name()

print("Cleaned calendar shape:", calendar_clean.shape)
display(calendar_clean.head())


Cleaned calendar shape: (1393570, 8)


,listing_id,date,available,price,month,month_name,year,day_of_week
0,241032,2016-01-04,True,85.0,1,January,2016,Monday
1,241032,2016-01-05,True,85.0,1,January,2016,Tuesday
2,241032,2016-01-06,False,NaN,1,January,2016,Wednesday
3,241032,2016-01-07,False,NaN,1,January,2016,Thursday
4,241032,2016-01-08,False,NaN,1,January,2016,Friday


## 6. Missing-Value Review

Missing values are inspected rather than blindly deleting every row. Some Airbnb fields are optional, so removing all rows containing any null value would unnecessarily discard useful records.


In [13]:
def missing_summary(df):
    result = pd.DataFrame({
        "Missing Values": df.isna().sum(),
        "Missing Percentage": (df.isna().mean() * 100).round(2)
    })
    return result[result["Missing Values"] > 0].sort_values(
        "Missing Percentage", ascending=False
    )

print("Listings missing values:")
display(missing_summary(listings_clean).head(20))

print("Reviews missing values:")
display(missing_summary(reviews_clean).head(20))

print("Calendar missing values:")
display(missing_summary(calendar_clean).head(20))


Listings missing values:


,Missing Values,Missing Percentage
square_feet,3721,97.46
monthly_price,2301,60.27
security_deposit,1952,51.13
weekly_price,1809,47.38
notes,1606,42.06
neighborhood_overview,1032,27.03
cleaning_fee,1030,26.98
transit,934,24.46
host_about,859,22.50
host_acceptance_rate,773,20.25


Reviews missing values:


,Missing Values,Missing Percentage
comments,18,0.02


Calendar missing values:


,Missing Values,Missing Percentage
price,459028,32.94


## 7. Exploratory Data Analysis


In [14]:
print("Total listings:", listings_clean["id"].nunique())
print("Average listing price:", round(listings_clean["price"].mean(), 2))
print("Total review records:", len(reviews_clean))

if "review_scores_rating" in listings_clean.columns:
    print("Average rating:", round(listings_clean["review_scores_rating"].mean(), 2))

print("Overall availability rate (%):",
      round(calendar_clean["available"].mean() * 100, 2))


Total listings: 3818
Average listing price: 127.98
Total review records: 84849
Average rating: 94.54
Overall availability rate (%): 67.06


In [15]:
# Average price by room type
room_type_price = (
    listings_clean.groupby("room_type", dropna=False)["price"]
    .mean()
    .sort_values(ascending=False)
)
display(room_type_price)


room_type
Entire home/apt    155.843369
Private room        75.044828
Shared room         47.547009
Name: price, dtype: float64

In [16]:
# Top neighbourhoods by average price
neighbourhood_price = (
    listings_clean.groupby("neighbourhood_cleansed")["price"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
)
display(neighbourhood_price)


neighbourhood_cleansed
Southeast Magnolia     231.705882
Portage Bay            227.857143
Westlake               194.470588
West Queen Anne        187.769231
Montlake               182.789474
Briarcliff             176.571429
Sunset Hill            176.055556
Industrial District    173.333333
Alki                   171.619048
Windermere             169.900000
Name: price, dtype: float64

In [17]:
# Monthly calendar price trend
monthly_price = (
    calendar_clean.groupby(["month", "month_name"])["price"]
    .mean()
    .reset_index()
    .sort_values("month")
)
display(monthly_price)


,month,month_name,price
0,1,January,122.912176
1,2,February,124.293927
2,3,March,128.644488
3,4,April,135.097005
4,5,May,139.538183
5,6,June,147.473137
6,7,July,152.094150
7,8,August,150.656594
8,9,September,143.255949
9,10,October,137.031939


In [18]:
# Monthly availability rate
monthly_availability = (
    calendar_clean.groupby(["month", "month_name"])["available"]
    .mean()
    .mul(100)
    .reset_index(name="availability_rate")
    .sort_values("month")
)
display(monthly_availability)


,month,month_name,availability_rate
0,1,January,56.693731
1,2,February,66.220805
2,3,March,70.918738
3,4,April,66.384669
4,5,May,67.567042
5,6,June,67.438449
6,7,July,62.709745
7,8,August,64.505145
8,9,September,67.440196
9,10,October,69.651397


In [19]:
# Average price by day of week in natural order
day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

day_price = (
    calendar_clean.groupby("day_of_week")["price"]
    .mean()
    .reindex(day_order)
)
display(day_price)


day_of_week
Monday       135.676414
Tuesday      135.408764
Wednesday    135.447880
Thursday     136.476032
Friday       143.036294
Saturday     143.202136
Sunday       136.459941
Name: price, dtype: float64

## 8. Validate Relationships for Power BI


In [20]:
listing_ids = set(listings_clean["id"].dropna())

calendar_match_rate = calendar_clean["listing_id"].isin(listing_ids).mean() * 100
reviews_match_rate = reviews_clean["listing_id"].isin(listing_ids).mean() * 100

print(f"Calendar listing_id match rate: {calendar_match_rate:.2f}%")
print(f"Reviews listing_id match rate: {reviews_match_rate:.2f}%")


Calendar listing_id match rate: 100.00%
Reviews listing_id match rate: 100.00%


## 9. Export Cleaned Datasets

The cleaned CSV files can be imported into MySQL and Power BI.

**Power BI relationships**
- `listings_clean[id]` → `calendar_clean[listing_id]` (1:*)
- `listings_clean[id]` → `reviews_clean[listing_id]` (1:*)


In [21]:
listings_clean.to_csv("listings_clean.csv", index=False)
reviews_clean.to_csv("reviews_clean.csv", index=False)
calendar_clean.to_csv("calendar_clean.csv", index=False)

print("Cleaned datasets exported successfully.")


Cleaned datasets exported successfully.


## 10. Analysis Outcome

The cleaned datasets support the three Power BI dashboard pages developed in this project:

1. **Airbnb Seattle Market Overview**
2. **Pricing & Seasonal Analysis**
3. **Reviews & Host Analysis**

The analysis focuses on market KPIs, neighbourhood and room-type pricing, seasonal trends, availability, property types, guest capacity, reviews, ratings, and Superhost characteristics.
